# Intégration Finale : RAG + Jury Fine-tuné

Ce notebook est le "cerveau" final de votre projet. Il combine :
1.  **RAG (Retrieval-Augmented Generation)** : Pour trouver les informations factuelles dans vos documents JSON (dans `rag/data`).
2.  **Modèle Fine-tuné** : Pour reformuler ces informations avec le comportement "Membre de Jury Exigeant" appris lors de l'entraînement.

---

In [ ]:
# 1. Installation des dépendances (Si nécessaire)
# Note : On installe à la fois les outils de RAG (langchain, faiss, sentence-transformers) et de modèle (bitsandbytes, peft)
%pip install -q transformers sentence-transformers faiss-cpu peft bitsandbytes accelerate

# ==========================================
# 🛑 CHECK GPU (CRITIQUE)
# ==========================================
import torch
print(f"CUDA disponible : {torch.cuda.is_available()}")
if not torch.cuda.is_available():
    raise RuntimeError("❌ AUCUN GPU DÉTECTÉ ! Le runtime Colab doit être en mode T4 GPU, sinon la RAM va exploser lors du chargement 4-bits. Changez le 'Runtime Type'.")
else:
    print(f"✅ GPU détecté : {torch.cuda.get_device_name(0)}")

In [ ]:
import os
import json
import torch
import numpy as np
import faiss
from pathlib import Path
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

# ==========================================
# CONFIGURATION DES CHEMINS (GOOGLE DRIVE)
# ==========================================
from google.colab import drive
drive.mount('/content/drive')

# 1. Où sont vos documents (JSON) ?
# On pointe vers le dossier Drive que vous avez créé
DATA_DIR = "/content/drive/MyDrive/Projet_CNRS/rag/data"

# 2. Où est votre modèle fine-tuné ?
# Idem, on pointe vers le modèle sur le Drive
ADAPTER_MODEL_DIR = "/content/drive/MyDrive/Projet_CNRS/fine_tuning/models/mon_modele_cnrs"

# 3. Modèle de base (le même que pour l'entraînement)
BASE_MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.2"

# ==========================================
# 🔍 DEBUG : Vérification
# ==========================================
import os
print(f"📍 Test accès DATA : {os.path.exists(DATA_DIR)}")
print(f"📍 Test accès MODELE : {os.path.exists(ADAPTER_MODEL_DIR)}")

## Étape 1 : Indexation des Documents (RAG)
Nous allons lire tous les fichiers `.json` dans le dossier `data/`, découper le texte en morceaux (chunks) et créer un index de recherche.

In [ ]:
# Paramètres de découpage
CHUNK_SIZE = 300  # Nombre de mots environ
OVERLAP = 50      # Chevauchement pour garder le contexte

def safe_get_text(obj):
    """Extrait le texte de n'importe quel structure JSON (dict, list, str)"""
    if isinstance(obj, str): return obj
    if isinstance(obj, list): return "\n".join([safe_get_text(x) for x in obj if x])
    if isinstance(obj, dict):
        # Priorité aux champs explicites
        for k in ["text", "content", "body", "description"]:
            if k in obj: return safe_get_text(obj[k])
        return "\n".join([safe_get_text(v) for v in obj.values()])
    return ""

def load_and_chunk_data(data_dir):
    dataset = []
    files = list(Path(data_dir).glob("*.json"))
    print(f"Lecture de {len(files)} fichiers JSON...")
    
    for file_path in files:
        try:
            with open(file_path, "r", encoding="utf-8") as f:
                data = json.load(f)
            
            text = safe_get_text(data)
            if not text: continue
            
            # Découpage basique
            words = text.split()
            for i in range(0, len(words), CHUNK_SIZE - OVERLAP):
                chunk = " ".join(words[i : i + CHUNK_SIZE])
                dataset.append({
                    "source": file_path.name,
                    "content": chunk
                })
        except Exception as e:
            print(f"Erreur sur {file_path.name}: {e}")
            
    return dataset

# Exécution
documents = load_and_chunk_data(DATA_DIR)
print(f"✅ Indexation terminée : {len(documents)} morceaux de texte prêts.")

In [ ]:
# Création de la base vectorielle (FAISS)
print("Chargement du modèle d'embedding (E5-Base)... Cette étape peut prendre un moment.")
embedder = SentenceTransformer("intfloat/multilingual-e5-base")

if documents:
    # On embedde tous les textes (prefix 'passage: ' pour E5)
    doc_texts = ["passage: " + d["content"] for d in documents]
    embeddings = embedder.encode(doc_texts, show_progress_bar=True, normalize_embeddings=True)
    
    # Index FAISS
    d = embeddings.shape[1]
    index = faiss.IndexFlatIP(d)
    index.add(embeddings)
    print(f"✅ Base vectorielle prête avec {index.ntotal} vecteurs.")
else:
    print("⚠️ AUCUN DOCUMENT TROUVÉ ! Vérifiez que vous avez mis des JSON dans rag/data.")

## Étape 2 : Chargement du Modèle (Hybride)
Nous chargeons Mistral 7B + Vos Adapters (le "Jury").

In [ ]:
import gc
import torch

# Nettoyage préventif de la RAM avant de charger le gros modèle
gc.collect()
torch.cuda.empty_cache()
print("🧹 RAM nettoyée avant chargement du modèle.")

from transformers import BitsAndBytesConfig

# Configuration 4-bit (pour que ça tourne sur GPU/CPU light)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

print("Chargement du modèle de base (Mistral)...")
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto"
)

print(f"Chargement de vos adapters depuis {ADAPTER_MODEL_DIR}...")
# 🛑 SECURITE DESACTIVEE : Si le modèle n'est pas là, ça PLANTE (au moins on est sûr)
model = PeftModel.from_pretrained(base_model, ADAPTER_MODEL_DIR)
print("✅ Modèle hybride (Base + Jury) chargé avec succès !")

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_NAME)
tokenizer.pad_token = tokenizer.unk_token

## Étape 3 : Le Moteur de Réponse (Inférence)
La fonction `generate_jury_response` fait le lien :
1.  Elle cherche les infos pertinentes dans FAISS.
2.  Elle construit un prompt strict.
3.  Elle génère la réponse avec votre modèle.

In [ ]:
from sentence_transformers import CrossEncoder

# Initialisation du Reranker (pour trier les documents pertinents)
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

def retrieve_context(query, k=5): # On en récupère plus (5) pour filtrer après
    if not documents: return ""
    
    # 1. Recherche vectorielle (FAISS)
    q_emb = embedder.encode(["query: " + query], normalize_embeddings=True)
    scores, indices = index.search(q_emb, k)
    
    # 2. Préparation pour le Reranking
    candidates = []
    for i, idx in enumerate(indices[0]):
        if idx == -1: continue
        doc = documents[idx]
        candidates.append(doc)

    if not candidates: return ""

    # 3. Reranking (Passage du Cross-Encoder)
    # On compare (Question, Document) pour avoir un score de pertinence précis
    pairs = [[query, doc["content"]] for doc in candidates]
    cross_scores = reranker.predict(pairs)
    
    # 4. Tri et Sélection du Top 3
    # On associe le score au document
    scored_candidates = sorted(zip(cross_scores, candidates), key=lambda x: x[0], reverse=True)
    top_docs = scored_candidates[:3] # On garde les 3 meilleurs après vérification rigoureuse
    
    # 5. Formatage
    retrieved_chunks = []
    print(f"\n🔍 [DEBUG] Documents trouvés pour : '{query}'")
    for score, doc in top_docs:
        print(f"   - (Score {score:.4f}) {doc['source']} : {doc['content'][:50]}...")
        retrieved_chunks.append(f"[Source: {doc['source']}]\n{doc['content']}")
        
    return "\n\n".join(retrieved_chunks)

def generate_jury_response(user_question):
    # 1. Retrieval
    context = retrieve_context(user_question)
    
    # 2. Construction du Prompt (System + Context + Ranking)
    system_prompt = (
        "Tu es un membre de jury de concours CNRS exigeant, précis et professionnel. "
        "Utilise STRICTEMENT les informations de CONTEXTE ci-dessous pour répondre. "
        "Si l'information n'est pas dans le contexte, dis-le clairement sans inventer.\n\n"
        f"### CONTEXTE DOCS OFFICIELS :\n{context}\n\n"
    )
    
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_question}
    ]
    
    prompt_str = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    
    # 3. Génération
    inputs = tokenizer(prompt_str, return_tensors="pt").to("cuda")
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=300,
            do_sample=True,
            temperature=0.4, # On baisse la température pour être plus "carré" et factuel
            pad_token_id=tokenizer.pad_token_id
        )
        
    response = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    return response

## Étape 4 : Testez votre Jury IA !
Posez vos questions ici.

In [ ]:
question = "Quelles sont les conditions pour le concours externe ingénieur d'étude ?"
reponse = generate_jury_response(question)

print(f"❓ Question : {question}")
print("-" * 50)
print(f"🤖 Jury : {reponse}")